# HY-489 Lab 4 — Imaging and DICOM

**Friday 13 November 2026 · 2 hours · practice, not graded**

Name(s): _______________________

Read `HANDOUT.md` alongside this. The handout has the reasoning; this notebook
is the session.

Lab 5 on 27 November is graded, it de-identifies this archive, and it starts
from the `inventory.csv` you write at the end of today.

**The rule, again.** Nothing real goes into this notebook, onto a public DICOM
server, or into this repository. Not your own images, not a friend's, not an
extract from a placement. The archive here is synthetic; `gen_archive.py` is
how it was made.

In [ ]:
import os, sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import checks, dicomio, image, inventory, reconcile

ARCHIVE = "../data/archive"
COHORT = "../../lab01/data/patients.csv"

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 44)
print("pydicom", __import__("pydicom").__version__)
print("files under the archive:", sum(len(f) for _, _, f in os.walk(ARCHIVE)))

---
## Part A · The archive

### Task 1 — look before you index

Open three files from three different folders and print all of them. Then
answer the three questions in the handout in the markdown cell below.

In [ ]:
import pydicom

ds = pydicom.dcmread(os.path.join(ARCHIVE, "incoming_2026_03", "img_0001.dcm"))
print(ds)

In [ ]:
# two more, from two other folders
for rel in ["incoming_2026_04/img_0050.dcm", "outside_referrals/img_0076.dcm"]:
    d = pydicom.dcmread(os.path.join(ARCHIVE, rel), stop_before_pixels=True)
    print(rel)
    print("   ", d.PatientID, "|", d.PatientName, "|", d.Modality,
          "|", d.get("StudyDescription", "<absent>"))

**Your answers.**

1. Which of PatientID, StudyInstanceUID, SeriesInstanceUID and SOPInstanceUID
   are human-readable?

   _write here_

2. Two tags that are present and empty. What does an empty tag mean that an
   absent tag does not?

   _write here_

3. How much of the Patient → Study → Series → Instance hierarchy can you
   recover from the file and folder names alone?

   _write here_

### Task 2 and 3 — `index_instance` and `build`

Both are in `src/inventory.py`. Edit the file, then re-run the cell below.
`%load_ext autoreload` is set up for you so you do not have to restart.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
rows, failures = inventory.build(ARCHIVE)
df = inventory.to_frame(rows)
print(len(rows), "indexed,", len(failures), "not")
for path, reason in failures:
    print("   ", path, "->", reason)

### Checkpoint 1 — 0:35

Six numbers. Say what each one is before you read it.

In [ ]:
inventory.summarise(df, failures)

**One pair of those numbers is not equal and should be.** Which pair, and what
does the inequality mean?

_write here_

### Task 4 — the four tables

Read them. Five things in here are wrong and you can see all five without
writing another line of code.

In [ ]:
inventory.patients(df)

In [ ]:
inventory.studies(df)

In [ ]:
inventory.series(df)

**Five things that are wrong, from the tables alone:**

1. _write here_
2. _write here_
3. _write here_
4. _write here_
5. _write here_

**One series has no `rows` and no `columns`.** It is not broken. What is it?

_write here_

### Task 5 — `check_patient_consistency`

In `src/checks.py`. Two questions: does every study belong to one patient, and
does every patient agree with themselves.

In [ ]:
checks.report(checks.check_patient_consistency(df))

### Task 6 — every check, including the slow one

In [ ]:
findings = []
findings += checks.check_unreadable(failures)
findings += checks.check_duplicate_sop(df)
findings += checks.check_patient_consistency(df)
findings += checks.check_photometric(df)
findings += checks.check_rescale(df)
findings += checks.check_pixel_spacing(df)
findings += checks.check_burned_in(df)
findings += checks.check_description_variants(df)
findings += checks.check_pixels_decode(ARCHIVE, df, dicomio.read_pixels)

n_findings = checks.report(findings)

**`check_pixels_decode` found something your `failures` list did not.**

What, and why did `build` miss it? One sentence, and it is the point of Part A.

_write here_

---
## Part B · Pixels and geometry

Three different numbers live in a DICOM image. Confusing them is the most
common imaging bug there is.

In [ ]:
image.hu_table()

### Task 1 and 2 — `to_real_values`, then measure the air

Fill `to_real_values` in `src/image.py`. Then measure the corner of every CT
series. Air is −1000 HU by definition, so the corner of a CT slice is a
calibration check that needs no tags at all.

In [ ]:
rowsB = []
for suid, g in df[df["modality"] == "CT"].groupby("series_uid"):
    if "LOCALIZER" in str(g["image_type"].iloc[0]):
        continue
    g = g.sort_values("instance_number")
    mid = g.iloc[len(g) // 2]
    ds, err = dicomio.read_pixels(os.path.join(ARCHIVE, mid["path"]))
    if err:
        continue
    real = image.to_real_values(ds)
    rowsB.append({
        "study": g["study_id"].iloc[0],
        "series": g["series_description"].iloc[0],
        "intercept": mid["rescale_intercept"],
        "corner air HU": round(image.corner_mean(real), 1),
        "min": real.min(), "max": real.max(),
    })
pd.DataFrame(rowsB)

**One series does not read about −1000.** Which one, how far off is it, and
what is missing from its header?

_write here_

The size of the error is not a coincidence. Say why it is that number.

_write here_

### Task 3 — `to_display`, and three windows of one slice

In [ ]:
ct = df[(df["modality"] == "CT") & (df["study_id"] == "S1")]
ct = ct[ct["series_description"] == "AXIAL 5MM"].sort_values("instance_number")
mid = ct.iloc[len(ct) // 2]
ds, _ = dicomio.read_pixels(os.path.join(ARCHIVE, mid["path"]))
real = image.to_real_values(ds)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
for ax, (name, c, w) in zip(axes, [("lung", -600, 1500),
                                   ("mediastinum", 50, 400),
                                   ("bone", 400, 1800)]):
    ax.imshow(image.to_display(ds, c, w), cmap="gray", vmin=0, vmax=1)
    ax.set_title("%s   C%d W%d" % (name, c, w))
    ax.axis("off")
plt.tight_layout(); plt.show()

### Task 3b — measure the tissues

The phantom was built with known values. Check them.

In [ ]:
size = real.shape[0]
print("%-14s %8.1f HU" % ("air, corners", image.corner_mean(real)))
for label, (cy, cx, r) in {
        "lung, left": (size * 0.50, size * 0.33, 7),
        "lung, right": (size * 0.50, size * 0.67, 7),
        "soft tissue": (size * 0.36, size * 0.50, 5),
        "spine": (size * 0.72, size * 0.50, 4)}.items():
    print("%-14s %8.1f HU" % (label, image.roi_mean(real, cy, cx, r)))

### Task 4 — the radiograph, both ways round

In [ ]:
cr = df[df["modality"] == "CR"].iloc[0]
ds_cr, _ = dicomio.read_pixels(os.path.join(ARCHIVE, cr["path"]))
print("PhotometricInterpretation:", dicomio.tag(ds_cr, "PhotometricInterpretation"))

naive = image.window(image.to_real_values(ds_cr),
                     float(dicomio.tag(ds_cr, "WindowCenter")),
                     float(dicomio.tag(ds_cr, "WindowWidth")))
correct = image.to_display(ds_cr)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 5))
axes[0].imshow(naive, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("photometric ignored"); axes[0].axis("off")
axes[1].imshow(correct, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("photometric honoured"); axes[1].axis("off")
plt.tight_layout(); plt.show()

**Which of the two would a person notice, and which would a program not?**

_write here_

### Task 5 — `check_slice_geometry`

In `src/checks.py`. Sort by `position_z`, difference the neighbours, compare
against the tags.

In [ ]:
geo = checks.check_slice_geometry(df)
checks.report(geo)

In [ ]:
# the raw spacings, to argue with the findings above
for suid, g in df[df["modality"] == "CT"].groupby("series_uid"):
    g = g[g["position_z"].notna()]
    if len(g) < 3:
        continue
    dz = checks.observed_spacing(g["position_z"].tolist())
    print("%-4s %-20s thickness=%-5s tag spacing=%-5s  gaps=%s"
          % (g["study_id"].iloc[0], str(g["series_description"].iloc[0])[:20],
             g["slice_thickness"].iloc[0], g["spacing_between_slices"].iloc[0],
             sorted(set(dz.tolist()))))

**A gap of exactly zero is a different problem from a gap of 10 mm.** Say what
each one is.

_write here_

### Task 6 — a distance in millimetres, and one that cannot be

In [ ]:
for label, row in [("CT", mid), ("MR", df[df["modality"] == "MR"].iloc[0])]:
    d, _ = dicomio.read_pixels(os.path.join(ARCHIVE, row["path"]))
    mm = image.distance_mm(d, (0, 0), (0, 40), dicomio.tag)
    print("%-3s  PixelSpacing=%-14s 40 pixels = %s"
          % (label, image.mm_per_pixel(d, dicomio.tag),
             "%.1f mm" % mm if mm is not None else "cannot be stated in mm"))

**Do not work around the `None`.** What would you have to do to report a
distance on that series, and who would have to tell you?

_write here_

### Checkpoint 2 — 1:20

Four things on screen: the air table with the odd series visible, the three
windows, the radiograph both ways, and your geometry findings.

---
## Part C · The archive against the cohort

The archive knows `PAGNI100275`. The CSV knows a UUID. Neither identifier is in
the other file.

In [ ]:
cohort = reconcile.load_cohort(COHORT)
pats = inventory.patients(df)
print("cohort rows:", len(cohort))
pats

### Task 2 — `match`, in `src/reconcile.py`

In [ ]:
results = reconcile.match(pats, cohort)
reconcile.report(results, pats)

### Checkpoint 3 — 1:50

Print, for every patient the rule refused, the archive row and the closest
cohort row one above the other.

In [ ]:
for r in results:
    if r["status"] == "exact":
        continue
    arc = pats.loc[r["patient_id"]]
    best = next(c for c in cohort if c["Id"] == r["best_cohort_id"])
    print("%-13s %-22s %-10s %-3s   archive"
          % (r["patient_id"], "/".join(arc["names"]),
             "/".join(arc["birth_dates"]), "/".join(arc["sexes"])))
    print("%-13s %-22s %-10s %-3s   closest cohort row"
          % (best["Id"][:12],
             "%s^%s" % (best["LAST"].upper(), best["FIRST"].upper()),
             best["BIRTHDATE"].replace("-", ""), best["GENDER"]))
    print("%-13s status=%s  agrees on %s  differs on %s"
          % ("", r["status"], ", ".join(r["agreed"]) or "nothing",
             ", ".join(r["disagreed"])))
    print()

**Two of those were refused outright. One is the same person as the row beneath
it and one is not.**

Which is which, and what did you use to tell them apart?

_write here_

Whatever you just used is the thing that does not scale. Say why.

_write here_

### The question to leave with

Nine archive patients, and the rule resolved six. Would you run a study on six,
on seven, or on nine? One sentence, and say what population each answer gives
you.

_write here_

---
## Wrap

Two files, both in this folder.

In [ ]:
df.to_csv("inventory.csv", index=False)
print("wrote inventory.csv:", len(df), "rows")

In [ ]:
report = []
report.append("# Lab 4 — archive report")
report.append("")
report.append("Name(s): ")
report.append("")
report.append("## The five numbers")
report.append("")
report.append("| | |")
report.append("|---|---|")
report.append("| files that read | %d |" % len(df))
report.append("| files that did not | %d |" % len(failures))
report.append("| patients | %d |" % df["patient_id"].nunique())
report.append("| studies | %d |" % df["study_uid"].nunique())
report.append("| series | %d |" % df["series_uid"].nunique())
report.append("| distinct instances | %d |" % df["sop_uid"].nunique())
report.append("")
report.append("## Findings")
report.append("")
for f in findings + geo:
    report.append("- **%s** (%s) — %s. %s" % (f.check, f.severity, f.where,
                                              f.detail))
report.append("")
report.append("## Reconciliation")
report.append("")
report.append("| PatientID | status | cohort id | note |")
report.append("|---|---|---|---|")
for r in sorted(results, key=lambda r: r["status"]):
    report.append("| %s | %s | %s | %s |"
                  % (r["patient_id"], r["status"], r["cohort_id"] or "-",
                     r["note"]))
report.append("")
report.append("## The two paragraphs")
report.append("")
report.append("**The MR series and its missing PixelSpacing.** TODO")
report.append("")
report.append("**Six, seven or nine patients.** TODO")
report.append("")

open("archive_report.md", "w", encoding="utf-8").write("\n".join(report))
print("wrote archive_report.md —  now replace the two TODOs by hand")

Replace the two TODOs in `archive_report.md` with the paragraphs you wrote
above. The table is generated; the two paragraphs are the part a person has to
write, and they are the part Lab 5 actually uses.

Then, from `labs/lab04/`:

```bash
python3 check_outputs.py
```